In [29]:
import pandas as pd
import dash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import dash_bootstrap_components as dbc


# =============================
# DATA PREPARATION
# =============================

df = pd.read_csv("data.csv")

df = df[
[
"CustomerID",
"Gender",
"Location",
"Product_Category",
"Quantity",
"Avg_Price",
"Transaction_Date",
"Month",
"Discount_pct"
]
]

df["CustomerID"] = df["CustomerID"].fillna(0).astype(int)

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Avg_Price"] = pd.to_numeric(df["Avg_Price"], errors="coerce")
df["Discount_pct"] = pd.to_numeric(df["Discount_pct"], errors="coerce")

df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"], errors="coerce")

df = df.dropna(subset=["Transaction_Date","Quantity","Avg_Price"])

df["Total_price"] = df["Quantity"] * df["Avg_Price"] * (1 - df["Discount_pct"]/100)

df["Week"] = df["Transaction_Date"].dt.isocalendar().week


# =============================
# APP
# =============================

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])


# =============================
# LAYOUT
# =============================

app.layout = html.Div([


# =============================
# HEADER
# =============================

html.Div([

html.Div([
html.H2("ECAP Store Dashboard", style={
"fontWeight":"bold",
"margin":"0",
"color":"white"
}),
html.P("Analyse des ventes",style={"color":"#e8f1f5","margin":"0"})
]),

dcc.Dropdown(
id="location_filter",
options=[{"label":i,"value":i} for i in sorted(df["Location"].unique())],
multi=True,
placeholder="Filtrer par zone",
style={
"width":"300px",
"background":"white",
"borderRadius":"8px"
}
)

],style={
"background":"linear-gradient(90deg,#4e73df,#224abe)",
"padding":"20px",
"display":"flex",
"justifyContent":"space-between",
"alignItems":"center",
"boxShadow":"0px 4px 12px rgba(0,0,0,0.15)"
}),



# =============================
# DASHBOARD BODY
# =============================

html.Div([


# =============================
# LEFT SIDE
# =============================

html.Div([


# KPI CARDS
html.Div([

dbc.Card([
dbc.CardBody([
html.P("Chiffre d'affaire - December", className="text-muted"),
html.H1(id="kpi_ca", style={"fontWeight":"bold"}),
html.Div(id="kpi_ca_var")
])
],style={"borderRadius":"15px","boxShadow":"0px 4px 10px rgba(0,0,0,0.08)","width":"48%"}),

dbc.Card([
dbc.CardBody([
html.P("Nombre de ventes - December", className="text-muted"),
html.H1(id="kpi_sales", style={"fontWeight":"bold"}),
html.Div(id="kpi_sales_var")
])
],style={"borderRadius":"15px","boxShadow":"0px 4px 10px rgba(0,0,0,0.08)","width":"48%"})

],style={
"display":"flex",
"justifyContent":"space-between",
"marginBottom":"20px"
}),


# BAR CHART CARD
dbc.Card([
dbc.CardBody([
dcc.Graph(id="bar_chart")
])
],style={
"borderRadius":"15px",
"boxShadow":"0px 4px 10px rgba(0,0,0,0.08)"
})


],style={
"width":"45%",
"padding":"20px"
}),



# =============================
# RIGHT SIDE
# =============================

html.Div([


# LINE CHART CARD
dbc.Card([
dbc.CardBody([
dcc.Graph(id="line_chart")
])
],style={
"borderRadius":"15px",
"boxShadow":"0px 4px 10px rgba(0,0,0,0.08)",
"marginBottom":"20px"
}),


# TABLE CARD
dbc.Card([
dbc.CardBody([

html.H5("Table des 100 dernières ventes",style={"marginBottom":"15px"}),

dash_table.DataTable(
id="sales_table",
page_size=10,
filter_action="native",
sort_action="native",

style_table={
"overflowX":"auto"
},

style_header={
"backgroundColor":"#4e73df",
"color":"white",
"fontWeight":"bold",
"textAlign":"center"
},

style_cell={
"textAlign":"center",
"padding":"8px",
"fontFamily":"Arial"
},

style_data={
"backgroundColor":"white",
"border":"1px solid #f1f1f1"
},

style_data_conditional=[
{
'if': {'row_index': 'odd'},
'backgroundColor': '#f8f9fc'
}
]
)

])
],style={
"borderRadius":"15px",
"boxShadow":"0px 4px 10px rgba(0,0,0,0.08)"
})


],style={
"width":"55%",
"padding":"20px"
})


],style={
"display":"flex",
"background":"#f5f7fb",
"minHeight":"100vh"
})

])


# =============================
# CALLBACK
# =============================

@app.callback(

[
Output("kpi_ca","children"),
Output("kpi_ca_var","children"),
Output("kpi_sales","children"),
Output("kpi_sales_var","children"),
Output("line_chart","figure"),
Output("bar_chart","figure"),
Output("sales_table","data"),
Output("sales_table","columns")
],

Input("location_filter","value")

)
def update_dashboard(location):

    dff = df.copy()

    if location:
        dff = dff[dff["Location"].isin(location)]

# =============================
# KPI
# =============================

    dec = dff[dff["Transaction_Date"].dt.month == 12]
    nov = dff[dff["Transaction_Date"].dt.month == 11]

    ca_dec = dec["Total_price"].sum()
    ca_nov = nov["Total_price"].sum()

    sales_dec = len(dec)
    sales_nov = len(nov)

    var_ca = ca_dec - ca_nov
    var_sales = sales_dec - sales_nov

    kpi_ca = f"{ca_dec/1000:.0f}k"
    kpi_sales = f"{sales_dec}"

    kpi_ca_var = html.Span(
        f"▲ {var_ca/1000:.0f}k" if var_ca >= 0 else f"▼ {abs(var_ca)/1000:.0f}k",
        style={"color": "green" if var_ca >= 0 else "red"}
    )

    kpi_sales_var = html.Span(
        f"▲ {var_sales}" if var_sales >= 0 else f"▼ {abs(var_sales)}",
        style={"color": "green" if var_sales >= 0 else "red"}
    )


# =============================
# LINE CHART
# =============================

    weekly = (
        dff.groupby(pd.Grouper(key="Transaction_Date", freq="W"))["Total_price"]
        .sum()
        .reset_index()
    )

    line = px.line(
        weekly,
        x="Transaction_Date",
        y="Total_price",
        title="Évolution du chiffre d'affaires par semaine",
        labels={
            "Transaction_Date": "Semaine",
            "Total_price": "Chiffre d'affaires"
        }
    )

    line.update_layout(
        template="plotly_white",
        title_x=0.5
    )

    line.update_xaxes(
        dtick="M3",
        tickformat="%b %Y",
        tickangle=0
    )


# =============================
# BAR CHART (NOMBRE DE COMMANDES)
# =============================

    # filtrer décembre (comme ton prof)
    dff_dec = dff[dff["Transaction_Date"].dt.month == 12]

    # nombre de commandes (IMPORTANT)
    df_grouped = (
        dff_dec.groupby(["Product_Category", "Gender"])
        .size()
        .unstack(fill_value=0)
    )

    # tri décroissant
    df_grouped["total"] = df_grouped.sum(axis=1)
    df_grouped = df_grouped.sort_values("total", ascending=False).head(10)
    df_grouped = df_grouped.drop(columns="total")

    # graphique
    bar = px.bar(
        df_grouped.reset_index().melt(id_vars="Product_Category"),
        x="value",
        y="Product_Category",
        color="Gender",
        orientation="h",
        barmode="group",
        title="Top 10 des catégories - Nombre de commandes (Décembre)",
        labels={
            "value": "Nombre de commandes",
            "Product_Category": "Catégorie",
            "Gender": "Genre"
        },
        color_discrete_map={
            "F": "#FFB6C1",
            "M": "#ADD8E6"
        },
        category_orders={
            "Product_Category": df_grouped.index.tolist()
        }
    )

    bar.update_layout(
        template="plotly_white",
        title_x=0.5,
        legend_title="Genre"
    )


# =============================
# TABLE
# =============================

    table = dff.sort_values("Transaction_Date", ascending=False).head(100).copy()

    table["Transaction_Date"] = table["Transaction_Date"].dt.strftime("%d/%m/%Y")
    table["Avg_Price"] = table["Avg_Price"].round(2)
    table["Total_price"] = table["Total_price"].round(2)

    table = table[
        [
            "Transaction_Date",
            "Gender",
            "Location",
            "Product_Category",
            "Quantity",
            "Avg_Price",
            "Discount_pct",
            "Total_price"
        ]
    ]

    columns = [
        {"name": "Date", "id": "Transaction_Date"},
        {"name": "Sexe", "id": "Gender"},
        {"name": "Ville", "id": "Location"},
        {"name": "Catégorie produit", "id": "Product_Category"},
        {"name": "Quantité", "id": "Quantity"},
        {"name": "Prix moyen (€)", "id": "Avg_Price"},
        {"name": "Remise (%)", "id": "Discount_pct"},
        {"name": "Prix total (€)", "id": "Total_price"}
    ]


# =============================
# RETURN
# =============================

    return (
        kpi_ca,
        kpi_ca_var,
        kpi_sales,
        kpi_sales_var,
        line,
        bar,
        table.to_dict("records"),
        columns
    )
# =============================
# RUN
# =============================

if __name__ == "__main__":
    app.run(debug=True,port=8026)